In [5]:
%%writefile vector_add.cu
#include <iostream>
using namespace std;


__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

int main() {
    int N;
    cout << "Enter size of vectors: ";
    cin >> N;

    int *A = new int[N];
    int *B = new int[N];
    int *C = new int[N];

    cout << "Enter elements of Vector A:\n";
    for (int i = 0; i < N; i++) {
        cin >> A[i];
    }

    cout << "Enter elements of Vector B:\n";
    for (int i = 0; i < N; i++) {
        cin >> B[i];
    }

    size_t bytes = N * sizeof(int);

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytes, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    add<<<blocks, threads>>>(d_A, d_B, d_C, N);

    cudaMemcpy(C, d_C, bytes, cudaMemcpyDeviceToHost);

    cout << "Result (A + B):\n";
    for (int i = 0; i < N; i++) {
        cout << C[i] << " ";
    }
    cout << endl;

    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting vector_add.cu


In [6]:
!nvcc vector_add.cu -o vector_add
!./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter size of vectors: 7
Enter elements of Vector A:
9 2 0 -8 14 50 7
Enter elements of Vector B:
1 -1 20 8 4 6 8
Result (A + B):
10 1 20 0 18 56 15 
